# Access Manager Getting Started Guide

This notebook walks you through Access Manager operations step by step, from authentication setup to cleanup.

## Why This Matters
- The focus is **Access Manager**: a single SDK issues, inspects, rotates, and revokes the API keys that govern who can call your models — plus admin, proxy, and usage-metric controls.
- A **single JWT bearer token authenticates every request** — you provide it securely when prompted.
- Enter your connection details once (you are prompted for only `BASE_URL` and the token), then reuse them across every section.
- Every key is built from a typed request model (`KeyCreateRequest`, `AdminKeyCreateRequest`, `KeyRegenerateRequest`), so each operation is explicit, validated, and easy to re-run.

## What You Will Accomplish
**Across the Access Manager surface (Sections 2–8):**
- Inspect the SDK with **`blueprint()`** and confirm the service is reachable with a **health check**.
- Manage the full **user key** lifecycle — list, create, inspect, regenerate, and revoke.
- Perform **admin** key operations across users, and drive the **LiteLLM admin proxy** (models, user info, spend).
- Review **usage metrics** by date range and model group, then **revoke** every key created during the run.


## Table of Contents

- [**1. Setup & Prerequisites**](#1-setup--prerequisites)
  - [1.1 Import Required Modules](#11-import-required-modules)
  - [1.2 Authentication](#12-authentication)
  - [1.3 Provide Connection Details](#13-provide-connection-details)
  - [1.4 Initialize Runtime Holders](#14-initialize-runtime-holders)
  - [1.5 Initialize Required Classes](#15-initialize-required-classes)
- [**2. Inspect Blueprint**](#2-inspect-blueprint)
- [**3. Health Check**](#3-health-check)
- [**4. Keys APIs**](#4-keys-apis)
  - [4.1 List User Keys](#41-list-user-keys)
  - [4.2 Create User Key](#42-create-user-key)
  - [4.3 Get User Key Details](#43-get-user-key-details)
  - [4.4 Create Regeneration Seed Key](#44-create-regeneration-seed-key)
  - [4.5 Regenerate User Key](#45-regenerate-user-key)
  - [4.6 Revoke User Key](#46-revoke-user-key)
- [**5. Admin APIs**](#5-admin-apis)
  - [5.1 List Keys Across Users](#51-list-keys-across-users)
  - [5.2 Create Admin Key](#52-create-admin-key)
  - [5.3 Get Admin Key Details](#53-get-admin-key-details)
  - [5.4 Regenerate Key as Admin](#54-regenerate-key-as-admin)
  - [5.5 Revoke Keys as Admin](#55-revoke-keys-as-admin)
- [**6. LiteLLM Admin Proxy**](#6-litellm-admin-proxy)
  - [6.1 Inspect Proxy Class](#61-inspect-proxy-class)
  - [6.2 Proxy GET Models](#62-proxy-get-models)
  - [6.3 Proxy GET User Info](#63-proxy-get-user-info)
  - [6.4 Proxy GET Model Info](#64-proxy-get-model-info)
  - [6.5 Proxy POST Reset Spend](#65-proxy-post-reset-spend)
  - [6.6 Add Model via Proxy](#66-add-model-via-proxy)
  - [6.7 Update Model via Proxy](#67-update-model-via-proxy)
  - [6.8 Delete Model via Proxy](#68-delete-model-via-proxy)
  - [6.9 Test Model Connection](#69-test-model-connection)
- [**7. Usage Metrics APIs**](#7-usage-metrics-apis)
  - [7.1 Overall Usage Activity](#71-overall-usage-activity)
  - [7.2 Model-Group Usage Activity](#72-model-group-usage-activity)
- [**8. Cleanup**](#8-cleanup)
  - [8.1 Revoke Keys Created in This Run](#81-revoke-keys-created-in-this-run)
- [**9. Data Models Reference**](#9-data-models-reference)
  - [9.1 Request Models Used in This Notebook](#91-request-models-used-in-this-notebook)
  - [9.2 Connection Inputs](#92-connection-inputs)

## 1. Setup & Prerequisites

This section prepares everything you need: imports, connection details, and the initialized client classes.

### 1.1 Import Required Modules

Run this first. It imports the Access Manager classes and request models, and defines `show_output()` so responses are easy to read.


In [6]:
from datetime import datetime
from getpass import getpass
from pprint import pprint

from teradata_agentstack import BearerAuth
from teradata_agentstack.access_manager import (
    AccessManagerClient,
    AdminKeys,
    ApiKeys,
    Health,
    LiteLLMProxy,
    UsageMetrics,
    blueprint,
)
from teradata_agentstack.access_manager.models import (
    AdminKeyCreateRequest,
    KeyCreateRequest,
    KeyRegenerateRequest,
)


def show_output(label, value):
    """Pretty-print an API response (Pydantic model or plain value)."""
    print(f'\n{label}:')
    formatter = getattr(value, 'model_dump', None) or getattr(value, 'dict', None) or (lambda: value)
    pprint(formatter() if callable(formatter) else formatter, sort_dicts=False, width=120)


def unique_alias(base):
    """Return an alias with a run timestamp suffix so it never collides with an existing key."""
    return f"{base}-{datetime.now():%Y%m%d-%H%M%S}"

### 1.2 Authentication

The `teradata_agentstack` Access Manager client supports **4 authentication modes** and **3 ways to provide credentials**.

The credential sources are resolved in this order: direct `auth` parameter, environment variables, then YAML config file.

#### Authentication Modes

| Auth Mode | Class | Required Fields |
| --- | --- | --- |
| Bearer Token | `BearerAuth` | `auth_bearer` |
| Basic Auth | `BasicAuth` | `username`, `password` |
| Client Credentials (OAuth2) | `ClientCredentialsAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret` |
| Device Code (OAuth2) | `DeviceCodeAuth` | `auth_token_url`, `auth_client_id`, `auth_client_secret`, `auth_device_auth_url` |

This notebook uses `BearerAuth` for Access Manager requests. You enter the token securely via `getpass` in step 1.3 — it is never hardcoded.

### 1.3 Provide Connection Details

Enter the Access Manager service URL when prompted &mdash; usually the only value you need to type. This cell also captures:

- `AUTH_TOKEN` &mdash; entered securely via `getpass` so it is never echoed or stored in the notebook.
- `SSL_VERIFY` &mdash; defaults to `False` for demo environments with self-signed certificates. Set `True` when your environment uses trusted certificates.

No credential values are printed &mdash; only whether each one was captured.

In [95]:
# Endpoint for the Access Manager service you are targeting.
BASE_URL = getpass('Enter Access Manager BASE_URL: ').strip()

# Bearer token (JWT) for the Access Manager service, entered securely.
AUTH_TOKEN = getpass('Enter AUTH_TOKEN: ').strip()

# Verify the endpoint's TLS certificate? Defaults to False for demo environments
# that use self-signed certificates. Set True when your environment uses trusted certs.
SSL_VERIFY = False

print('Connection details captured.')
print(f'BASE_URL configured:   {bool(BASE_URL)}')
print(f'AUTH_TOKEN configured: {bool(AUTH_TOKEN)}')
print(f'SSL_VERIFY:            {SSL_VERIFY}')

Connection details captured.


### 1.4 Initialize Runtime Holders

This cell defines the four key-id placeholders shared across cells: `USER_KEY_ID`, `REGENERATE_KEY_ID`, `ADMIN_KEY_ID`, and `TARGET_ADMIN_USER_ID`. Later cells fill them in as keys are created, and the cleanup step uses them to revoke whatever this run created.



In [4]:
# --- Runtime holders (filled in as the notebook runs; consumed by later cells & cleanup) ---
USER_KEY_ID          = None
REGENERATE_KEY_ID    = None
ADMIN_KEY_ID         = None
TARGET_ADMIN_USER_ID = None

print('Runtime settings ready. Per-cell demo values are defined inline in each section.')


Runtime settings ready. Per-cell demo values are defined inline in each section.


### 1.5 Initialize Required Classes

This creates the authenticated Access Manager client and the helper classes (`ApiKeys`, `AdminKeys`, `Health`, `LiteLLMProxy`, `UsageMetrics`) so the later examples can run directly.


In [97]:
auth = BearerAuth(auth_bearer=AUTH_TOKEN)
client = AccessManagerClient(base_url=BASE_URL, auth=auth, ssl_verify=SSL_VERIFY)

keys = ApiKeys(client=client)
admin = AdminKeys(client=client)
health = Health(client=client)
proxy = LiteLLMProxy(client=client)
usage_metrics = UsageMetrics(client=client)

print('Access Manager classes initialized.')


Access Manager classes initialized.


## 2. Inspect Blueprint

A quick discovery step. It prints the blueprint so you can see the available classes and methods before making API calls.


In [3]:
blueprint()

----------------------------------------------------------------
Available classes for API Access Manager SDK:
    * teradata_agentstack.access_manager.AdminKeys
    * teradata_agentstack.access_manager.Health
    * teradata_agentstack.access_manager.ApiKeys
    * teradata_agentstack.access_manager.LiteLLMProxy
    * teradata_agentstack.access_manager.UsageMetrics
----------------------------------------------------------------


## 3. Health Check

Run these checks first to confirm the service is reachable and healthy. If these fail, fix connectivity before running write operations.

In [99]:
show_output('Health Check', health.health_check())
show_output('Liveness', health.liveness())
show_output('Readiness', health.readiness())
show_output('Metrics', health.metrics())


Health Check:
{'status': 'healthy',
 'timestamp': '2026-06-18T12:31:39.134674403Z',
 'components': {'background_jobs': {'status': 'healthy',
                                    'details': {'jobs_enabled': True, 'service_running': True},
                                    'last_check': '2026-06-18T12:31:39.134687703Z'},
                'database': {'status': 'healthy',
                             'response_time_ms': 1,
                             'details': {'active_keys_count': 51,
                                         'idle_connections': 1,
                                         'in_use_connections': 0,
                                         'max_idle_connections': 0,
                                         'max_open_connections': 25,
                                         'open_connections': 1,
                                         'wait_count': 0,
                                         'wait_duration_ms': 0},
                             'last_check': '2026-06-18T

## 4. Keys APIs

This section shows the full user key lifecycle: list, create, inspect, rotate, and revoke. Treat create/regenerate/revoke steps as data-changing operations.

### 4.1 List User Keys

A read-only check that returns your current keys, so you can confirm what already exists before creating new ones.


In [100]:
list_result = keys.list(limit=50, offset=0)
show_output('List User Keys', list_result)


List User Keys:
{'keys': [{'key_id': '<id>',
           'alias': 'my-sdk-demo-key-rotated-20260618-045715',
           'status': 'revoked',
           'expires_at': '2026-07-18T11:58:10.136123Z',
           'created_at': '2026-06-18T11:58:10.26642Z',
           'metadata': {'created_via': 'sdk', 'purpose': 'development', 'tags': ['sdk-demo', 'regen']}},
          {'key_id': '<id>',
           'alias': 'my-sdk-demo-key-20260618-045624',
           'status': 'revoked',
           'expires_at': '2026-07-18T11:57:54.371646Z',
           'created_at': '2026-06-18T11:57:54.471377Z',
           'metadata': {'created_via': 'sdk', 'purpose': 'development', 'tags': ['sdk-demo']}},
          {'key_id': '<id>',
           'alias': 'my-sdk-demo-key-rotated-20260618-023247',
           'status': 'revoked',
           'expires_at': '2026-07-18T10:27:17.537139Z',
           'created_at': '2026-06-18T10:27:17.670644Z',
           'metadata': {'created_via': 'sdk', 'purpose': 'development', 'tags': ['s

### 4.2 Create User Key

Creates a new user key and stores its `key_id` in `USER_KEY_ID` for reuse in later cells. This operation changes data.


In [101]:
create_key_body = KeyCreateRequest(
    duration='P30D',                                   # ISO-8601 duration (max 90 days)
    key_alias=unique_alias('my-sdk-demo-key'),         # run-timestamp suffix keeps the alias unique
    metadata={'purpose': 'development', 'tags': ['sdk-demo'], 'created_via': 'sdk'},
    models=['gpt-4', 'gpt-3.5-turbo'],                 # omit / empty list = no model access
    tpm_limit=18789,
    rpm_limit=5,
)
created_key = keys.create(body=create_key_body)
show_output('Created User Key', created_key)

USER_KEY_ID = getattr(created_key, 'key_id', None)
print(f'USER_KEY_ID: {USER_KEY_ID}')



Created User Key:
{'key_id': '<id>',
 'api_key': '<REDACTED>',
 'alias': 'my-sdk-demo-key-20260618-053017',
 'status': 'active',
 'expires_at': '2026-07-18T12:31:47.796449296Z',
 'created_at': '2026-06-18T12:31:47.939069197Z'}
USER_KEY_ID: <id>


### 4.3 Get User Key Details

Fetches full details for one key, including status, expiry, allowed models, and rate limits. Use it to verify key configuration.


In [102]:
show_output('Get User Key Details', keys.get(id=USER_KEY_ID))



Get User Key Details:
{'key_id': '<id>',
 'user_id': '<REDACTED_EMAIL>',
 'api_key': '<REDACTED>',
 'alias': 'my-sdk-demo-key-20260618-053017',
 'status': 'active',
 'expires_at': '2026-07-18T12:31:47.796449Z',
 'created_at': '2026-06-18T12:31:47.935759Z',
 'updated_at': '2026-06-18T12:31:47.783777Z',
 'metadata': {'created_via': 'sdk', 'purpose': 'development', 'tags': ['sdk-demo']}}


### 4.4 Create Regeneration Seed Key

Creates a separate key to use in the regenerate example and saves it to `REGENERATE_KEY_ID`. This operation changes data.


In [103]:
regen_seed_body = KeyCreateRequest(
    duration='P30D',
    key_alias=unique_alias('my-sdk-demo-key-regen'),   # run-timestamp suffix keeps the alias unique
    metadata={'purpose': 'development', 'tags': ['sdk-demo', 'regen'], 'created_via': 'sdk'},
    models=['gpt-4', 'gpt-3.5-turbo'],
    tpm_limit=18789,
    rpm_limit=5,
)
regen_seed_key = keys.create(body=regen_seed_body)
show_output('Created Regeneration Seed Key', regen_seed_key)

REGENERATE_KEY_ID = getattr(regen_seed_key, 'key_id', None)
print(f'REGENERATE_KEY_ID: {REGENERATE_KEY_ID}')



Created Regeneration Seed Key:
{'key_id': '<id>',
 'api_key': '<REDACTED>',
 'alias': 'my-sdk-demo-key-regen-20260618-053023',
 'status': 'active',
 'expires_at': '2026-07-18T12:31:53.9792203Z',
 'created_at': '2026-06-18T12:31:54.133092603Z'}
REGENERATE_KEY_ID: <id>


### 4.5 Regenerate User Key

Rotates an existing key while keeping the key record. The previous key value becomes invalid immediately after regeneration.


In [104]:
regenerate_body = KeyRegenerateRequest(key_alias=unique_alias('my-sdk-demo-key-rotated'))
regenerated_key = keys.regenerate(id=REGENERATE_KEY_ID, body=regenerate_body)
show_output('Regenerated User Key', regenerated_key)



Regenerated User Key:
{'key_id': '<id>',
 'api_key': '<REDACTED>',
 'alias': 'my-sdk-demo-key-rotated-20260618-053026',
 'status': 'active',
 'regenerated_at': '2026-06-18T12:31:57.253152341Z',
 'expires_at': '2026-07-18T12:31:53.97922Z'}


### 4.6 Revoke User Key

Permanently revokes a key so it can no longer authenticate requests. Use carefully because this action cannot be undone.


In [105]:
show_output('Revoke User Key', keys.revoke(id=USER_KEY_ID))
USER_KEY_ID = None



Revoke User Key:
{'key_id': '<id>',
 'message': 'API key successfully revoked',
 'revoked_at': '2026-06-18T12:32:00Z',
 'status': 'revoked'}


## 5. Admin APIs

Manage keys for other users across your tenant (requires admin privileges).


### 5.1 List Keys Across Users

A read-only, tenant-wide view of all keys. Useful for an audit before making admin changes.


In [ ]:
show_output('Admin - List Keys Across Users', admin.list(limit=50, offset=0))


### 5.2 Create Admin Key

Creates a key for the user in `user_email` and stores its id in `ADMIN_KEY_ID` for the next steps. Edit `user_email` to a real user before running.



In [107]:
# Set user_email to the email of the user the admin key is created for.
admin_create_body = AdminKeyCreateRequest(
    user_email='user@example.com',
    duration='P90D',
    metadata={'purpose': 'development', 'tags': ['sdk-admin'], 'created_via': 'sdk'},
    key_alias=unique_alias('admin-provisioned-key'),   # run-timestamp suffix keeps the alias unique
    models=['anthropic-claude-3-5-haiku-20241022-v1-0-profile'],
    max_budget=50.0,
)
admin_created = admin.create(body=admin_create_body)
show_output('Admin Created Key', admin_created)

ADMIN_KEY_ID = getattr(admin_created, 'key_id', None)
TARGET_ADMIN_USER_ID = getattr(admin_created, 'user_id', None)
print(f'ADMIN_KEY_ID: {ADMIN_KEY_ID}')
print(f'TARGET_ADMIN_USER_ID: {TARGET_ADMIN_USER_ID}')



Admin Created Key:
{'key_id': '<id>',
 'api_key': '<REDACTED>',
 'alias': 'admin-provisioned-key-20260618-053035',
 'status': 'active',
 'expires_at': '2026-09-16T12:32:06.34419229Z',
 'created_at': '2026-06-18T12:32:06.454320201Z'}
ADMIN_KEY_ID: <id>
TARGET_ADMIN_USER_ID: None


### 5.3 Get Admin Key Details

Retrieves details for an admin-managed key so you can validate ownership, limits, and current status.


In [108]:
show_output('Admin Get Key Details', admin.get(id=ADMIN_KEY_ID))



Admin Get Key Details:
{'key_id': '<id>',
 'key_alias': 'admin-provisioned-key-20260618-053035',
 'user_id': '<REDACTED_EMAIL>',
 'user_email': '<REDACTED_EMAIL>',
 'info': {'access_group_ids': [],
          'aliases': {},
          'allowed_cache_controls': [],
          'allowed_routes': [],
          'auto_rotate': False,
          'blocked': None,
          'budget_duration': None,
          'budget_id': None,
          'budget_reset_at': None,
          'config': {},
          'created_at': '2026-06-18T12:32:06.377000+00:00',
          'created_by': 'default_user_id',
          'expires': '2026-09-16T12:32:06.344192Z',
          'key_alias': 'admin-provisioned-key-20260618-053035',
          'key_name': '<REDACTED>',
          'key_rotation_at': None,
          'last_active': None,
          'last_rotation_at': None,
          'litellm_budget_table': None,
          'litellm_organization_table': None,
          'litellm_project_table': None,
          'max_budget': 50,
          

### 5.4 Regenerate Key as Admin

Rotates a user's key using admin access. The admin regenerate API is **user-id based**, so this cell sets `TARGET_ADMIN_USER_ID` to the target user. Change it to the real user the key belongs to before running. This operation changes data.



In [109]:
TARGET_ADMIN_USER_ID= 'user@example.com'
admin_regen_body = KeyRegenerateRequest(key_alias=unique_alias('admin-provisioned-key-rotated'))
admin_regen = admin.regenerate(user_id=TARGET_ADMIN_USER_ID, id=ADMIN_KEY_ID, body=admin_regen_body)
show_output('Admin Regenerated Key', admin_regen)



Admin Regenerated Key:
{'key_id': '<id>',
 'api_key': '<REDACTED>',
 'alias': 'admin-provisioned-key-rotated-20260618-053041',
 'status': 'active',
 'regenerated_at': '2026-06-18T12:32:12.543388851Z',
 'expires_at': '2026-09-16T12:32:06.344192Z'}


### 5.5 Revoke Keys as Admin

Shows two admin revoke options: revoke one key, then revoke all keys for a user. Both operations are permanent.


In [110]:
show_output('Admin Revoke Specific Key', admin.revoke(user_id=TARGET_ADMIN_USER_ID, id=ADMIN_KEY_ID))
show_output('Admin Revoke All User Keys', admin.revoke_user_keys(user_id=TARGET_ADMIN_USER_ID))
ADMIN_KEY_ID = None



Admin Revoke Specific Key:
{'action': 'admin_revoke',
 'key_id': '<id>',
 'message': 'API key successfully revoked',
 'revoked_at': '2026-06-18T12:32:15Z',
 'revoked_by': '<REDACTED_EMAIL>',
 'revoked_by_email': '<REDACTED_EMAIL>',
 'user_id': '<REDACTED_EMAIL>'}

Admin Revoke All User Keys:
{'action': 'bulk_revoke',
 'message': 'All user keys successfully revoked',
 'revoked_at': '2026-06-18T12:32:15Z',
 'revoked_by': '<REDACTED_EMAIL>',
 'revoked_by_email': '<REDACTED_EMAIL>',
 'user_id': '<REDACTED_EMAIL>'}


## 6. LiteLLM Admin Proxy

Common proxy endpoints called with `roles='ADMIN'` via the helper methods (`get`, `post`, `patch`).

> Read-only calls run as-is. The add, update, delete, and connection-test cells are **optional** &mdash; each stays skipped until you fill in the required value (a provider API key, or a model id to delete).



### 6.1 Inspect Proxy Class

Displays available `LiteLLMProxy` methods and docstrings so you can understand supported operations before running them.


In [111]:
help(LiteLLMProxy)

Help on class LiteLLMProxy in module teradata_agentstack.access_manager:

class LiteLLMProxy(builtins.object)
 |  LiteLLMProxy(client)
 |
 |  Wildcard proxy that enforces regex and role policies before forwarding requests to LiteLLM with the master key. Inference endpoints are blocked.
 |
 |  Methods defined here:
 |
 |  __init__ = _constructor(self, client) from teradata_agentstack._utils
 |      Constructor for the dynamic class.
 |      :param client: The client instance to be used by the class.
 |
 |  delete(*c, **kwargs) from teradata_agentstack._utils._create_dynamic_method.<locals>
 |      DESCRIPTION:
 |          The function 'delete' does the following:
 |          - Wildcard proxy - enforces regex+role policies; forwards to LiteLLM with master key. Inference endpoints are blocked.
 |
 |      PARAMETERS:
 |          body (Optional):
 |              JSON body forwarded to LiteLLM
 |              Types: dict
 |
 |          roles (Required):
 |              Comma-separated roles


### 6.2 Proxy GET Models

Read-only call to list registered proxy models and confirm current routing entries.


In [ ]:
show_output('Proxy GET /models', proxy.get(proxy_path='models', roles='ADMIN'))

### 6.3 Proxy GET User Info

Fetches current proxy user details, including budgets and limits, to validate your admin context.


In [ ]:
show_output('Proxy GET /user/info', proxy.get(proxy_path='user/info', roles='ADMIN'))

### 6.4 Proxy GET Model Info

Retrieves model metadata, provider configuration, and usage-related details for registered models.


In [ ]:
show_output('Proxy GET /model/info', proxy.get(proxy_path='model/info', roles='ADMIN'))

### 6.5 Proxy POST Reset Spend

Resets proxy spend counters. Use only in non-production or controlled test scenarios. This operation changes system state.


In [115]:
show_output('Proxy POST /global/spend/reset', proxy.post(proxy_path='global/spend/reset', roles='ADMIN', body={}))


Proxy POST /global/spend/reset:
{'message': 'Spend for all API Keys and Teams reset successfully', 'status': 'success'}


### 6.6 Add Model via Proxy

Creates a new proxy model entry with provider connection settings. Keep provider secrets secure and avoid hardcoding them.


In [118]:
# Enter a provider API key to register a model; press Enter to skip this cell.
provider_api_key = getpass('Enter provider API key (press Enter to skip): ').strip()
if provider_api_key:
    add_model_body = {
        'model_name': 'demo-model',
        'litellm_params': {
            'model': 'gpt-4o',
            'custom_llm_provider': 'openai',
            'api_base': 'http://localhost:9801',                # optional provider endpoint
            'api_key': provider_api_key,
        },
        'model_info': {},
    }
    show_output('Proxy POST /model/new', proxy.post(proxy_path='model/new', roles='ADMIN', body=add_model_body))
else:
    print('Add Model via Proxy: skipped (no provider API key entered).')



Proxy POST /model/new:
{'model_id': '<id>',
 'model_name': 'demo-model',
 'litellm_params': {'model': 'gpt-4o',
                    'api_key': '<REDACTED>',
                    'api_base': 'http://localhost:9801',
                    'use_litellm_proxy': False,
                    'custom_llm_provider': 'openai',
                    'use_in_pass_through': False,
                    'merge_reasoning_content_in_choices': False},
 'model_info': {'id': '<id>', 'db_model': False},
 'created_at': '2026-06-18T12:36:42.647000Z',
 'created_by': 'default_user_id',
 'updated_at': '2026-06-18T12:36:42.647000Z',
 'updated_by': 'default_user_id'}


### 6.7 Update Model via Proxy

Updates an existing proxy model configuration (for example provider model or connection fields). This operation changes data.


In [121]:
# Enter a provider API key to update the model; press Enter to skip this cell.
provider_api_key = getpass('Enter provider API key (press Enter to skip): ').strip()
if provider_api_key:
    update_model_body = {
        'model_name': 'demo-model',
        'litellm_params': {
            'model': 'gpt-4o',
            'custom_llm_provider': 'openai',
            'api_base': '',
            'api_key': provider_api_key,
        },
        'model_info': {'mode': 'chat'},
    }
    show_output('Proxy PATCH /model/update', proxy.patch(proxy_path='model/update', roles='ADMIN', body=update_model_body))
else:
    print('Update Model via Proxy: skipped (no provider API key entered).')


HTTPError: Error message: {"error":"forbidden","message":"policy denies this operation"}

### 6.8 Delete Model via Proxy

Removes a model entry from the proxy by id. Any traffic mapped to that model will fail after deletion.


In [120]:
# Enter a model id to delete; press Enter to skip this cell.
model_id_to_delete = input('Enter model id to delete (press Enter to skip): ').strip()
if model_id_to_delete:
    show_output('Proxy POST /model/delete', proxy.post(proxy_path='model/delete', roles='ADMIN', body={'id': model_id_to_delete}))
else:
    print('Delete Model via Proxy: skipped (no model id entered).')



Proxy POST /model/delete:
{'message': 'Model: <id> deleted successfully'}


### 6.9 Test Model Connection

Validates provider connectivity through the proxy using the supplied `litellm_params` before production use.


In [122]:
# Enter a provider API key to test connectivity; press Enter to skip this cell.
provider_api_key = getpass('Enter provider API key (press Enter to skip): ').strip()
if provider_api_key:
    test_connection_body = {
        'litellm_params': {
            'model': 'gpt-4o',
            'custom_llm_provider': 'openai',
            'api_base': '',
            'api_key': provider_api_key,
        },
        'model_info': {},
    }
    show_output('Proxy POST /health/test_connection', proxy.post(proxy_path='health/test_connection', roles='ADMIN', body=test_connection_body))
else:
    print('Test Model Connection: skipped (no provider API key entered).')



Proxy POST /health/test_connection:
{'status': 'error',
 'result': {'api_base': '',
            'custom_llm_provider': 'openai',
            'use_in_pass_through': False,
            'use_litellm_proxy': False,
            'merge_reasoning_content_in_choices': False,
            'model': 'gpt-4o',
            'litellm_metadata': {'tags': ['litellm-internal-health-check'],
                                 'user_api_key_hash': 'litellm-internal-health-check',
                                 'user_api_key_alias': 'litellm-internal-health-check',
                                 'user_api_key_spend': 0.0,
                                 'user_api_key_max_budget': None,
                                 'user_api_key_team_id': 'litellm-internal-health-check',
                                 'user_api_key_project_id': None,
                                 'user_api_key_user_id': None,
                                 'user_api_key_org_id': None,
                                 'user_api

## 7. Usage Metrics APIs

Use these calls to review usage trends over time by date range, then drill down by model group when needed.

### 7.1 Overall Usage Activity

Fetches aggregate usage activity between the `start_date` and `end_date` passed to the call. The example uses a fixed range (`2026-06-01` to `2026-06-13`) &mdash; edit these inline values to match the window you want to inspect.



In [123]:
show_output('Overall Activity', usage_metrics.overall_activity(start_date='2026-06-01', end_date='2026-06-13'))



Overall Activity:
{'start_date': '2026-06-01',
 'end_date': '2026-06-13',
 'totals': {'spend': 0.0063032,
            'prompt_tokens': 1559993,
            'completion_tokens': 103816,
            'cache_read_input_tokens': 0,
            'cache_creation_input_tokens': 0,
            'total_tokens': 1663809,
            'successful_requests': 616,
            'failed_requests': 15,
            'api_requests': 631},
 'daily': [{'date': '2026-06-04',
            'metrics': {'spend': 0,
                        'prompt_tokens': 8,
                        'completion_tokens': 12,
                        'cache_read_input_tokens': 0,
                        'cache_creation_input_tokens': 0,
                        'total_tokens': 20,
                        'successful_requests': 1,
                        'failed_requests': 0,
                        'api_requests': 1}},
           {'date': '2026-06-09',
            'metrics': {'spend': 0.0016573,
                        'prompt_tokens': 3

### 7.2 Model-Group Usage Activity

Fetches usage metrics filtered by one model group (for example `gpt-4` or `claude`) for targeted analysis.


In [124]:
show_output(
    'Model-Group Activity',
    usage_metrics.model_group_activity(
        start_date='2026-06-01',
        end_date='2026-06-13',
        model_group='us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile',
        model=None,
    ),
)



Model-Group Activity:
{'start_date': '2026-06-01',
 'end_date': '2026-06-13',
 'input': {'model_group': 'us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile'},
 'resolved': {'model_group': 'us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile',
              'resolution': 'merged',
              'candidates': ['bedrock/converse/<arn>',
                             'converse/<arn>',
                             'us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile'],
              'normalized': 'us-anthropic-claude-sonnet-4-5-20250929-v1-0-profile',
              'dimension': 'model_groups'},
 'totals': {'spend': 0,
            'prompt_tokens': 1490840,
            'completion_tokens': 97680,
            'cache_read_input_tokens': 0,
            'cache_creation_input_tokens': 0,
            'total_tokens': 1588520,
            'successful_requests': 384,
            'failed_requests': 10,
            'api_requests': 394},
 'daily': [{'date': '2026-06-04',
            'metrics': {'spen

## 8. Cleanup

Use this section to revoke keys created during the demo so your environment is left clean after testing.

### 8.1 Revoke Keys Created in This Run

Attempts cleanup using values captured earlier in the notebook. If required values are missing, the corresponding cleanup step is skipped safely.



In [125]:
for label, key_id in (('User Key', USER_KEY_ID), ('Regeneration Seed Key', REGENERATE_KEY_ID)):
    if key_id:
        show_output(f'Cleanup - Revoke {label}', keys.revoke(id=key_id))
    else:
        print(f'Cleanup - Revoke {label}: skipped (nothing to revoke).')

if ADMIN_KEY_ID and TARGET_ADMIN_USER_ID:
    show_output('Cleanup - Revoke Admin Key', admin.revoke(user_id=TARGET_ADMIN_USER_ID, id=ADMIN_KEY_ID))
else:
    print('Cleanup - Revoke Admin Key: skipped (nothing to revoke).')
USER_KEY_ID = REGENERATE_KEY_ID = ADMIN_KEY_ID = None
print('Cleanup complete.')


Cleanup - Revoke User Key: skipped (nothing to revoke).

Cleanup - Revoke Regeneration Seed Key:
{'key_id': '<id>',
 'message': 'API key successfully revoked',
 'revoked_at': '2026-06-18T12:38:52Z',
 'status': 'revoked'}
Cleanup - Revoke Admin Key: skipped (nothing to revoke).
Cleanup complete.


## 9. Data Models Reference

Use this quick reference to understand the request models and connection inputs.

### 9.1 Request Models Used in This Notebook

| API Area | Request Model | Purpose |
|----------|---------------|---------|
| User Keys | `KeyCreateRequest` | Create a user API key |
| User/Admin Regeneration | `KeyRegenerateRequest` | Rotate an existing key |
| Admin Keys | `AdminKeyCreateRequest` | Create a key for another user |

### 9.2 Connection Inputs

All three inputs are configured in step 1.3.

| Input | How it is provided | Description |
|-------|--------------------|-------------|
| `AUTH_TOKEN` | Prompted via `getpass` | Bearer token attached to every request |
| `BASE_URL` | Prompted via `getpass` | Access Manager service base URL &mdash; the only value you must enter |
| `SSL_VERIFY` | Set in code | Verify TLS certificates (defaults to `False`; set `True` for trusted certs) |

Last validated: 2026-06-22